# S4.2 — Text Chunker (Section-Aware)

Interactive verification of the section-aware text chunker.

**Key features:**
- Word-based chunking with sliding window (600 words, 100 overlap)
- Section-based chunking using PDF parser output
- Header prepending (title + abstract) for standalone context
- Metadata filtering (authors, affiliations, abstract duplicates)
- Graceful fallback from section-based to word-based

In [1]:
import sys
sys.path.insert(0, "../..")

from src.schemas.indexing import ChunkMetadata, TextChunk
from src.schemas.pdf import Section
from src.services.indexing.text_chunker import TextChunker

print("\u2713 Imports successful")

✓ Imports successful


## 1. Data Models

In [2]:
meta = ChunkMetadata(
    chunk_index=0, start_char=0, end_char=500,
    word_count=100, overlap_with_previous=0, overlap_with_next=50,
    section_title="Introduction"
)
chunk = TextChunk(
    text="Sample chunk text", metadata=meta,
    arxiv_id="2401.12345", paper_id="uuid-abc"
)
assert chunk.metadata.section_title == "Introduction"
assert chunk.arxiv_id == "2401.12345"
print(f"\u2713 TextChunk model works: chunk_index={meta.chunk_index}, section={meta.section_title}")

✓ TextChunk model works: chunk_index=0, section=Introduction


## 2. Word-Based Chunking

In [3]:
chunker = TextChunker(chunk_size=600, overlap_size=100, min_chunk_size=100)

# Generate a 1500-word text
text = " ".join(f"word{i}" for i in range(1500))
chunks = chunker.chunk_text(text, "2401.00001", "pid-1")

print(f"Input: {len(text.split())} words")
print(f"Output: {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"  Chunk {i}: {c.metadata.word_count} words, "
          f"overlap_prev={c.metadata.overlap_with_previous}, "
          f"overlap_next={c.metadata.overlap_with_next}")

assert len(chunks) == 3
assert chunks[0].metadata.overlap_with_previous == 0
assert chunks[0].metadata.overlap_with_next == 100
print("\n\u2713 Word-based chunking works correctly")

Input: 1500 words
Output: 3 chunks
  Chunk 0: 600 words, overlap_prev=0, overlap_next=100
  Chunk 1: 600 words, overlap_prev=100, overlap_next=100
  Chunk 2: 500 words, overlap_prev=100, overlap_next=0

✓ Word-based chunking works correctly


## 3. Section-Based Chunking

In [4]:
sections = [
    Section(title="Introduction", content=" ".join(f"intro{i}" for i in range(300)), level=1),
    Section(title="Methods", content=" ".join(f"method{i}" for i in range(200)), level=1),
    Section(title="Results", content=" ".join(f"result{i}" for i in range(1000)), level=1),
    Section(title="Conclusion", content=" ".join(f"conclusion{i}" for i in range(50)), level=1),
]

chunks = chunker.chunk_paper(
    title="Attention Is All You Need",
    abstract="We propose a new network architecture based on attention mechanisms.",
    full_text="fallback text",
    arxiv_id="1706.03762",
    paper_id="pid-transformer",
    sections=sections,
)

print(f"Sections: {[s.title for s in sections]}")
print(f"Output: {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"  Chunk {i}: section={c.metadata.section_title}, "
          f"{c.metadata.word_count} words")

assert len(chunks) >= 4  # Intro + Methods + Results (split) + Conclusion
assert "Attention Is All You Need" in chunks[0].text
assert "Abstract:" in chunks[0].text
print("\n\u2713 Section-based chunking works with header prepending")

Sections: ['Introduction', 'Methods', 'Results', 'Conclusion']
Output: 4 chunks
  Chunk 0: section=Introduction, 318 words
  Chunk 1: section=Methods, 218 words
  Chunk 2: section=Results (Part 1), 616 words
  Chunk 3: section=Results (Part 2) + Combined, 570 words

✓ Section-based chunking works with header prepending


## 4. Section Filtering

In [5]:
# Test that metadata sections are filtered
sections_with_metadata = {
    "Authors": "John Doe, Jane Smith",
    "Introduction": " ".join(f"word{i}" for i in range(200)),
    "Abstract": "We propose a new network architecture based on attention mechanisms.",
}

filtered = chunker._filter_sections(
    sections_with_metadata,
    "We propose a new network architecture based on attention mechanisms."
)

print(f"Input sections: {list(sections_with_metadata.keys())}")
print(f"After filtering: {list(filtered.keys())}")

assert "Authors" not in filtered
assert "Abstract" not in filtered
assert "Introduction" in filtered
print("\n\u2713 Metadata and abstract duplicate filtering works")

Input sections: ['Authors', 'Introduction', 'Abstract']
After filtering: ['Introduction']

✓ Metadata and abstract duplicate filtering works


## 5. Fallback Behavior

In [6]:
# No sections → word-based fallback
text = " ".join(f"word{i}" for i in range(600))
chunks = chunker.chunk_paper(
    title="Test", abstract="Abstract",
    full_text=text, arxiv_id="2401.99999", paper_id="pid-fb",
    sections=None,
)

assert len(chunks) == 1
assert chunks[0].metadata.section_title is None
print(f"No sections: {len(chunks)} chunk(s), word-based fallback")

# Empty text → empty list
chunks_empty = chunker.chunk_paper(
    title="Test", abstract="Abstract",
    full_text="", arxiv_id="2401.00000", paper_id="pid-empty",
    sections=None,
)
assert chunks_empty == []
print("Empty text: 0 chunks")

print("\n\u2713 Fallback behavior verified")

Empty text provided for paper 2401.00000


No sections: 1 chunk(s), word-based fallback
Empty text: 0 chunks

✓ Fallback behavior verified


## 6. Configuration Validation

In [7]:
try:
    TextChunker(chunk_size=100, overlap_size=100)
    assert False, "Should have raised ValueError"
except ValueError as e:
    print(f"Caught expected error: {e}")

print("\n\u2713 Configuration validation works")

Caught expected error: Overlap size must be less than chunk size

✓ Configuration validation works
